# Pipeline completo Orion: curva de luz → secundaria → diagramas HR

Reúne en un solo notebook lo que antes estaba repartido en
`1_obtencion_priors_curva`, `2_obtencion_priors_model`,
`2_teff_desde_isocronas` y `3_diagramas_hr_isocronas`:

1. **Priors de la curva de luz**: por default se usa `priors_v2.csv`, que tiene las
   profundidades medidas en el fondo del eclipse (`1b_profundidades_eclipse_v2.ipynb`, donde
   está la justificación). Opcionalmente se puede recalcular BLS + Lomb-Scargle sobre todos
   los archivos de `Light_Curve_Files/`.
2. **Primaria** (por TIC y por rama MIST/Somers17): Teff, L, masa y edad del ajuste
   massage. R1 sale de Stefan-Boltzmann.
3. **Secundaria**: se usa `Depth_primary = (R2/R1)^2`, es decir, `R2 = R1·sqrt(Depth_primary)`.
   Con R2 y la edad de la rama se obtienen M2, Teff2 y L2 contra la malla de la
   propia rama y contra BHAC15.
4. **Combinación de sectores** por TIC (promedio pesado de la profundidad).
5. **Diagramas**: HR y R–masa por TIC, y HR por sector.

### Qué cambia respecto a los notebooks anteriores

- **Una sola celda de configuración** (rutas, TICs a excluir, ramas, flags). No hay
  archivos intermedios entre etapas: todo corre en memoria y al final se guarda en
  `resultados_pipeline/`, así que no se pisa nada de lo anterior.
- **Interpolación de isocronas más robusta.** Antes se usaba `LinearNDInterpolator`
  sobre puntos dispersos en (logAge, masa). En MIST las masas van de 0.1 a 169 Msun,
  así que los triángulos salen muy deformes y aparecían cientos de avisos de
  "fuera del casco convexo". Ahora:
  - En mallas de *isocronas* (MIST, BHAC15) se interpola cada isocrona en log(masa
    inicial) y luego linealmente en logAge entre las dos edades vecinas.
  - En mallas de *tracks* (Somers17) se interpola cada track en logAge.
  - La isocrona de cada edad se calcula **una sola vez y se guarda en caché**, así
    que las secundarias, las incertidumbres y las curvas de las figuras la reutilizan.
- **R→M a edad fija** se resuelve directo sobre esa isocrona (el primer cruce desde
  masas bajas, igual que antes), sin `brentq` ni escaneos de 400 puntos por estrella.
- **Formato largo**: una fila por (TIC, Sector, rama, malla). La tabla ancha
  compatible con `resumen_diagramas_hr_por_tic.csv` se arma a partir de ella.
- **Se usa una única relación R2(R1, Depth)**. `2_obtencion_priors_model` usaba
  `R2 = R1·sqrt(1 − Depth)`, que no concuerda con los otros notebooks.
- **Nuevo**: una secundaria por TIC combinando los sectores, con un indicador de
  inconsistencia entre sectores (χ²_red).
- La etapa de curvas de luz sigue la lógica de `Orion_yceb/priors_chidos.ipynb` (v3),
  que es la que produjo `priors_completosv1.csv`: BLS en dos etapas con duración
  adaptativa y máscara con salvaguarda. Ahora corre en lote sobre todos los archivos.

**Lo único que normalmente hay que tocar es la celda de configuración.**

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

pd.set_option("display.max_columns", 60)

## 0) Configuración

In [ ]:
# ============================================================
# RUTAS
# ============================================================
RUTA_ORION = Path("/home/luciam/Documents/starry/Resultados_Orion")
RUTA_MODELOS = RUTA_ORION / "datos_estrellas_massage_orion" / "modelos"
RUTA_CURVAS = RUTA_ORION / "Light_Curve_Files"

PATH_CATALOGO = RUTA_ORION / "catalogo_completo_mist_sommers.csv"   # ajuste massage (_1 MIST, _2 Somers17)
PATH_PRIORS = RUTA_ORION / "priors_v2.csv"   # profundidades v2 (1b_profundidades_eclipse_v2.ipynb)

ARCHIVOS_MALLAS = {
    "MIST": RUTA_MODELOS / "iso_mist" / "MIST_iso_6a9a687177c08_todas_edades.csv",
    "Somers17": RUTA_MODELOS / "iso_somers17" / "f017track_ms.csv",
    "BHAC15": RUTA_MODELOS / "iso_barafe" / "BHAC15_iso.txt",
}

SALIDA = RUTA_ORION / "resultados_pipeline"   # todo lo que genera este notebook va aquí

# ============================================================
# ETAPA 1: CURVAS DE LUZ
# ============================================================
# False -> usa PATH_PRIORS tal cual.
# True  -> recalcula los priors de todos los archivos en RUTA_CURVAS y usa esos
#          (se guardan en SALIDA/priors_recalculados.csv, PATH_PRIORS no se toca).
RECALCULAR_PRIORS = False

# ventana de wotan por archivo, si alguno necesita una distinta al default
VENTANA_WOTAN_POR_ARCHIVO = {
    # "tic 138829403_S_6_LC.txt": 0.50,
}

# ============================================================
# SELECCION Y MODELOS
# ============================================================
# 2_obtencion_priors_model excluía [11412481, 264459857, 264483228, 138829403];
# los notebooks 2_teff/3_diagramas no excluían ninguno.
TICS_EXCLUIR = []

# rama = de qué ajuste massage salen Teff1, L1, M1 y la edad del sistema
RAMAS = {"MIST": "1", "Somers17": "2"}   # nombre -> sufijo de columnas en el catálogo
ORDEN_RAMAS = ("MIST", "Somers17")       # prioridad al elegir "la" primaria/secundaria de cada TIC
MALLA_SECUNDARIA = "BHAC15"              # malla usada para la secundaria "elegida" (cubre hasta 0.01 Msun)

# False = como en 3_diagramas: sigma_R2 solo lleva el error de la profundidad.
# True  = también propaga sigma_R1 (de T_std y lgL_std).
PROPAGAR_ERROR_R1 = False

CHI2_RED_MAX = 3.0        # arriba de esto, las profundidades de los sectores de un TIC se marcan inconsistentes
MOSTRAR_FIGURAS = True    # False -> solo se guardan en disco
T_SUN = 5772.0            # K

SALIDA.mkdir(parents=True, exist_ok=True)

## 1) Priors desde la curva de luz (opcional)

Misma lógica que `priors_chidos.ipynb` (v3), compactada y en lote:
wotan (biweight) → BLS en dos pasadas con grid de duración que se ensancha si
el óptimo toca el borde → P o 2P → profundidad primaria (fase 0) y secundaria
(fase 0.5) → máscara de eclipses con salvaguarda → Lomb-Scargle sobre lo que
queda fuera de eclipse.

Solo corre si `RECALCULAR_PRIORS = True`.

**Importante:** aquí la profundidad se mide con el estimador **v1** (mediana en una ventana
fija de ±0.03 en fase), que subestima eclipses angostos y en V. Si recalculas, pasa el
`priors_recalculados.csv` resultante por `1b_profundidades_eclipse_v2.ipynb` (cambiando
`PATH_PRIORS_V1`) antes de usarlo.

In [ ]:
CFG_CURVA = dict(
    time_column="#Time", flux_column="Flux_CBV_SS",
    window_length=0.55, break_tolerance=0.5,
    min_period_bls=0.3, max_period_bls=15.0,
    n_periods_coarse=3000, n_periods_fine=4000, fine_zoom_frac=0.02,
    duration_min_frac=0.005, duration_max_frac=0.12, n_durations=40,
    duration_grow_factor=1.6, max_duration_grid_tries=4,
    width_factor=3.0, max_mask_fraction=0.7, min_points_out_of_eclipse=30,
    min_period_ls=0.2, max_period_ls=15.0, n_freq_ls=5000,
    eclipse_half_width=0.03, out_of_eclipse_lo=0.15, out_of_eclipse_hi=0.35,
)


def cargar_curva(path, time_column, flux_column):
    sep = "," if str(path).lower().endswith(".csv") else r"\s+"
    df = pd.read_csv(path, sep=sep, engine="python")
    df.columns = [c.strip() for c in df.columns]
    t, f = df[time_column].to_numpy(float), df[flux_column].to_numpy(float)
    ok = np.isfinite(t) & np.isfinite(f)
    t, idx = np.unique(t[ok], return_index=True)   # ordena y quita tiempos duplicados
    f = f[ok][idx]
    if len(t) < 50:
        raise ValueError(f"Muy pocos puntos válidos ({len(t)}) en {path}")
    return t, f


def tic_sector_de_nombre(path):
    m = re.search(r"tic\s*0*(\d+).*?_S_0*(\d+)", Path(path).name, re.IGNORECASE)
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)


def _bls(bls, pmin, pmax, n, cfg):
    # ensancha el grid de duración mientras el óptimo quede pegado al techo
    frac_max = cfg["duration_max_frac"]
    periodos = np.linspace(pmin, pmax, n)
    for intento in range(1, cfg["max_duration_grid_tries"] + 1):
        dur = np.linspace(cfg["duration_min_frac"], frac_max, cfg["n_durations"]) * cfg["min_period_bls"]
        res = bls.power(periodos, dur)
        i = np.argmax(res.power)
        en_techo = np.isclose(res.duration[i], dur[-1], rtol=1e-6)
        if not en_techo:
            break
        frac_max *= cfg["duration_grow_factor"]
    return res, i, intento, en_techo


def analizar_curva(path, **kwargs):
    from astropy.timeseries import BoxLeastSquares, LombScargle
    from wotan import flatten

    cfg = {**CFG_CURVA, **kwargs}
    t, f = cargar_curva(path, cfg["time_column"], cfg["flux_column"])
    f_orb, tendencia = flatten(t, f, method="biweight", window_length=cfg["window_length"],
                               break_tolerance=cfg["break_tolerance"], return_trend=True)
    ok = np.isfinite(f_orb)
    t_o, f_o = t[ok], f_orb[ok]

    # -------- BLS: pasada gruesa + zoom --------
    bls = BoxLeastSquares(t_o, f_o)
    grueso, ig, _, _ = _bls(bls, cfg["min_period_bls"], cfg["max_period_bls"], cfg["n_periods_coarse"], cfg)
    P0 = grueso.period[ig]
    lo = max(cfg["min_period_bls"], P0 * (1 - cfg["fine_zoom_frac"]))
    hi = min(cfg["max_period_bls"], P0 * (1 + cfg["fine_zoom_frac"]))
    fino, i, intentos, en_techo = _bls(bls, lo, hi, cfg["n_periods_fine"], cfg)
    P_bls, t0, dur = fino.period[i], fino.transit_time[i], fino.duration[i]

    # -------- P o 2P: ¿los mínimos en fase 0 y 0.5 (con 2P) son distintos? --------
    fase2 = ((t_o - t0) / (2 * P_bls)) % 1
    es_2P = abs(np.nanmedian(f_o[np.abs(fase2) < 0.05]) - np.nanmedian(f_o[np.abs(fase2 - 0.5) < 0.05])) > 0.005
    P_orb = 2 * P_bls if es_2P else P_bls

    # -------- profundidades --------
    fase = ((t_o - t0) % P_orb) / P_orb
    fuera = (fase > cfg["out_of_eclipse_lo"]) & (fase < cfg["out_of_eclipse_hi"])

    def profundidad(centro):
        dentro = np.abs(((fase - centro + 0.5) % 1) - 0.5) < cfg["eclipse_half_width"]
        if dentro.sum() < 5 or fuera.sum() < 5:
            return np.nan, np.nan
        d = np.nanmedian(f_o[fuera]) - np.nanmedian(f_o[dentro])
        return float(d), float(np.nanstd(f_o[dentro]) / np.sqrt(dentro.sum()))

    (d1, d1e), (d2, d2e) = profundidad(0.0), profundidad(0.5)

    # -------- máscara de eclipses (se angosta si tapa demasiado) --------
    wf = cfg["width_factor"]
    fase_t = ((t - t0) / P_orb) % 1
    for _ in range(6):
        ancho = min(wf * dur / P_orb, 0.5)
        mascara = (fase_t < ancho) | (fase_t > 1 - ancho) | (np.abs(fase_t - 0.5) < ancho)
        if mascara.mean() <= cfg["max_mask_fraction"]:
            break
        wf *= 0.7
    fallback = (~mascara).sum() < cfg["min_points_out_of_eclipse"]
    t_r, f_r = (t, f) if fallback else (t[~mascara], f[~mascara])

    # -------- rotación --------
    freq = np.linspace(1 / cfg["max_period_ls"], 1 / cfg["min_period_ls"], cfg["n_freq_ls"])
    P_rot = 1 / freq[np.argmax(LombScargle(t_r, f_r).power(freq))]

    tic, sector = tic_sector_de_nombre(path)
    fila = dict(
        TIC=tic, Sector=sector, P_bls=P_bls, P_orb=P_orb, is_2P=es_2P, P_rot=P_rot,
        t0=t0, Duration=dur,
        Depth_primary=d1, Depth_primary_err=d1e, Depth_secondary=d2, Depth_secondary_err=d2e,
        Mask_fraction=mascara.mean(), Width_factor_used=wf, Fallback_used=fallback,
        Duration_grid_tries=intentos, Duration_hit_ceiling=en_techo,
        window_length=cfg["window_length"], archivo=Path(path).name,
    )
    curvas = dict(t=t, f=f, tendencia=tendencia, t_o=t_o, f_o=f_o)
    return fila, curvas


def figura_curva(fila, curvas, savepath=None):
    t, f, tend, t_o, f_o = (curvas[k] for k in ("t", "f", "tendencia", "t_o", "f_o"))
    fase_orb = ((t_o - fila["t0"]) / fila["P_orb"]) % 1
    fase_rot = ((t - t.min()) / fila["P_rot"]) % 1

    fig, ax = plt.subplots(2, 2, figsize=(13, 8))
    ax[0, 0].plot(t, f, ".", ms=1, alpha=0.3, color="0.4")
    ax[0, 0].plot(t, tend, lw=1.5, color="tab:orange")
    ax[0, 0].set(title="(a) Detrending (wotan)", xlabel="Tiempo [d]", ylabel="Flujo")
    ax[0, 1].plot(t_o, f_o, ".", ms=1.5, alpha=0.5, color="tab:purple")
    ax[0, 1].set(title="(b) Curva aplanada", xlabel="Tiempo [d]", ylabel="Flujo normalizado")
    ax[1, 0].plot(fase_orb, f_o, ".", ms=1.5, alpha=0.4, color="tab:purple")
    ax[1, 0].set(title=f"(c) Fase orbital  P_orb={fila['P_orb']:.5f} d"
                       + ("  (2×P_bls)" if fila["is_2P"] else "")
                       + f"\nprof. primaria={fila['Depth_primary']*100:.3f}%",
                 xlabel="Fase", ylabel="Flujo normalizado")
    ax[1, 1].plot(fase_rot, tend, ".", ms=1.5, alpha=0.4, color="tab:blue")
    ax[1, 1].set(title=f"(d) Fase rotacional  P_rot={fila['P_rot']:.5f} d", xlabel="Fase", ylabel="Flujo")
    avisos = []
    if fila["Duration_hit_ceiling"]:
        avisos.append("duración BLS en el techo del grid")
    if fila["Fallback_used"]:
        avisos.append("LS con curva completa (máscara tapaba casi todo)")
    fig.suptitle(f"TIC {fila['TIC']}  Sector {fila['Sector']}"
                 + (f"   [!] {'; '.join(avisos)}" if avisos else ""), fontsize=12)
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.show() if MOSTRAR_FIGURAS else None
    plt.close(fig)


def procesar_curvas(carpeta, ventanas=None, figuras_en=None):
    ventanas = ventanas or {}
    filas = []
    for path in sorted(Path(carpeta).glob("*_LC.txt")):
        kw = {"window_length": ventanas[path.name]} if path.name in ventanas else {}
        try:
            fila, curvas = analizar_curva(path, **kw)
        except Exception as e:
            print(f"[x] {path.name}: {e}")
            continue
        filas.append(fila)
        print(f"[ok] {path.name}: P_orb={fila['P_orb']:.5f} d  Depth_primary={fila['Depth_primary']:.4f}")
        if figuras_en is not None:
            figura_curva(fila, curvas, Path(figuras_en) / f"TIC_{fila['TIC']}_S{fila['Sector']}_curva.png")
    return pd.DataFrame(filas).sort_values(["TIC", "Sector"]).reset_index(drop=True)

In [ ]:
if RECALCULAR_PRIORS:
    dir_fig_curvas = SALIDA / "curvas_de_luz"
    dir_fig_curvas.mkdir(exist_ok=True)
    priors = procesar_curvas(RUTA_CURVAS, VENTANA_WOTAN_POR_ARCHIVO, figuras_en=dir_fig_curvas)
    priors.to_csv(SALIDA / "priors_recalculados.csv", index=False)
    print(f"Guardado: {SALIDA / 'priors_recalculados.csv'}")
else:
    priors = pd.read_csv(PATH_PRIORS)
    print(f"Priors leídos de {PATH_PRIORS.name}")

priors = priors.loc[~priors["TIC"].isin(TICS_EXCLUIR)].reset_index(drop=True)
sin_depth = priors.loc[priors["Depth_primary"].isna(), ["TIC", "Sector"]]
print(f"{len(priors)} filas (TIC, Sector), {priors['TIC'].nunique()} TICs")
if len(sin_depth):
    print("Sin Depth_primary (no generan secundaria):", sin_depth.to_dict("records"))

## 2) Mallas de isocronas

Cada malla se carga en un formato estándar: `logAge, m_ini, mass, logTeff, logR, logL`.
`m_ini` es la coordenada que se usa para interpolar a lo largo de una isocrona (en MIST
es la masa inicial, que es monótona a lo largo de la isocrona; `mass` es la masa
actual).

`Malla.isocrona(edad_Myr)` devuelve la isocrona a esa edad y la guarda en caché.
`Malla.en_radio(R, edad_Myr)` invierte R→M sobre esa isocrona. Fuera de la cobertura
de la malla se devuelve NaN, nunca una extrapolación.

In [ ]:
COLS = ["mass", "logTeff", "logR", "logL"]


def cargar_mist(path):
    t = pd.read_csv(path, usecols=["log10_isochrone_age_yr", "initial_mass", "star_mass",
                                   "log_Teff", "log_R", "log_L"])
    return pd.DataFrame({"logAge": t["log10_isochrone_age_yr"], "m_ini": t["initial_mass"],
                         "mass": t["star_mass"], "logTeff": t["log_Teff"],
                         "logR": t["log_R"], "logL": t["log_L"]})


def cargar_somers17(path):
    t = pd.read_csv(path, usecols=["Mass", "Age(Gyr)", "log(Teff)", "log(R/Rsun)", "log(L/Lsun)"])
    return pd.DataFrame({"logAge": np.log10(t["Age(Gyr)"] * 1e9), "m_ini": t["Mass"],
                         "mass": t["Mass"], "logTeff": t["log(Teff)"],
                         "logR": t["log(R/Rsun)"], "logL": t["log(L/Lsun)"]})


def cargar_bhac15(path):
    # bloques "!  t (Gyr) = ..." seguidos de filas: M/Ms Teff L/Ls g R/Rs ...
    filas, edad = [], None
    with open(path) as fh:
        for linea in fh:
            s = linea.strip()
            if s.startswith("!  t"):
                edad = float(s.split("=")[1])
            elif edad is not None and s[:1].isdigit():
                v = s.split()
                filas.append((np.log10(edad * 1e9), float(v[0]), np.log10(float(v[1])),
                              float(v[2]), np.log10(float(v[4]))))
    t = pd.DataFrame(filas, columns=["logAge", "m_ini", "logTeff", "logL", "logR"])
    t["mass"] = t["m_ini"]
    return t


class Malla:
    # tipo="isocronas": pocas edades, muchas masas (MIST, BHAC15)
    # tipo="tracks":    pocas masas, muchas edades (Somers17)

    def __init__(self, nombre, tabla, tipo):
        self.nombre, self.tipo = nombre, tipo
        tabla = tabla.replace([np.inf, -np.inf], np.nan).dropna(subset=["logAge", "m_ini"] + COLS)
        clave, orden = ("logAge", "m_ini") if tipo == "isocronas" else ("m_ini", "logAge")
        self._grupos = {k: g.sort_values(orden).drop_duplicates(orden).reset_index(drop=True)
                        for k, g in tabla.groupby(clave)}
        self._claves = np.array(sorted(self._grupos))
        self.logAge_min, self.logAge_max = tabla["logAge"].min(), tabla["logAge"].max()
        self.edad_min, self.edad_max = 10 ** self.logAge_min / 1e6, 10 ** self.logAge_max / 1e6
        self._cache = {}
        print(f"[{nombre}] {tipo}: edad {self.edad_min:.3g}-{self.edad_max:.3g} Myr, "
              f"masa {tabla['mass'].min():.3f}-{tabla['mass'].max():.3f} Msun")

    # ---------------- isocrona a una edad ----------------
    def isocrona(self, edad_myr):
        if not (np.isfinite(edad_myr) and edad_myr > 0):
            return None
        logAge = round(float(np.log10(edad_myr * 1e6)), 6)
        if logAge not in self._cache:
            if not (self.logAge_min <= logAge <= self.logAge_max):
                self._cache[logAge] = None
            elif self.tipo == "isocronas":
                self._cache[logAge] = self._desde_isocronas(logAge)
            else:
                self._cache[logAge] = self._desde_tracks(logAge)
        return self._cache[logAge]

    def _desde_isocronas(self, logAge):
        edades = self._claves
        j = int(np.searchsorted(edades, logAge))
        if j < len(edades) and np.isclose(edades[j], logAge, atol=1e-6):
            return self._grupos[edades[j]][["m_ini"] + COLS]
        a0, a1 = edades[j - 1], edades[j]
        g0, g1 = self._grupos[a0], self._grupos[a1]
        w = (logAge - a0) / (a1 - a0)
        lo = max(g0["m_ini"].min(), g1["m_ini"].min())
        hi = min(g0["m_ini"].max(), g1["m_ini"].max())
        m = np.union1d(g0["m_ini"], g1["m_ini"])
        m = m[(m >= lo) & (m <= hi)]
        if len(m) < 2:
            return None
        lm = np.log10(m)
        out = {"m_ini": m}
        for c in COLS:
            v0 = np.interp(lm, np.log10(g0["m_ini"]), g0[c])
            v1 = np.interp(lm, np.log10(g1["m_ini"]), g1[c])
            out[c] = (1 - w) * v0 + w * v1
        return pd.DataFrame(out)

    def _desde_tracks(self, logAge):
        filas = []
        for m, g in self._grupos.items():
            if g["logAge"].iloc[0] <= logAge <= g["logAge"].iloc[-1]:
                filas.append([m] + [np.interp(logAge, g["logAge"], g[c]) for c in COLS])
        return pd.DataFrame(filas, columns=["m_ini"] + COLS) if len(filas) >= 2 else None

    # ---------------- inversión R -> M a edad fija ----------------
    def en_radio(self, R, edad_myr):
        # Propiedades del modelo con radio R a esa edad (primer cruce desde masas bajas).
        nada = dict(M=np.nan, Teff=np.nan, logL=np.nan)
        iso = self.isocrona(edad_myr)
        if iso is None or not (np.isfinite(R) and R > 0):
            return nada
        d = iso["logR"].to_numpy() - np.log10(R)
        cruces = np.nonzero(np.sign(d[:-1]) * np.sign(d[1:]) <= 0)[0]
        if len(cruces) == 0:
            return nada
        i = cruces[0]
        u = 0.0 if d[i] == d[i + 1] else d[i] / (d[i] - d[i + 1])
        val = lambda c: iso[c].iat[i] + u * (iso[c].iat[i + 1] - iso[c].iat[i])
        return dict(M=val("mass"), Teff=10 ** val("logTeff"), logL=val("logL"))


mallas = {
    "MIST": Malla("MIST", cargar_mist(ARCHIVOS_MALLAS["MIST"]), "isocronas"),
    "Somers17": Malla("Somers17", cargar_somers17(ARCHIVOS_MALLAS["Somers17"]), "tracks"),
    "BHAC15": Malla("BHAC15", cargar_bhac15(ARCHIVOS_MALLAS["BHAC15"]), "isocronas"),
}

## 3) Primaria: una fila por (TIC, rama)

Se usan las columnas `_median` como valor central y las `_std` como incertidumbre.
`R1 = sqrt(L/Lsun)·(T_sun/Teff)^2` y
`σ_R1/R1 = sqrt[(ln10/2·σ_logL)^2 + (2·σ_T/T)^2]`.

In [ ]:
def radio_sb(Teff, logL):
    return 10 ** (logL / 2) * (T_SUN / Teff) ** 2


def lum_sb(Teff, R):
    return R ** 2 * (Teff / T_SUN) ** 4


cat = pd.read_csv(PATH_CATALOGO)
cat = cat.loc[~cat["TIC"].isin(TICS_EXCLUIR)].reset_index(drop=True)

partes = []
for rama, s in RAMAS.items():
    p = pd.DataFrame({
        "TIC": cat["TIC"], "rama": rama,
        "Teff1": cat[f"T_median_{s}"], "Teff1_err": cat[f"T_std_{s}"],
        "logL1": cat[f"lgL_median_{s}"], "logL1_err": cat[f"lgL_std_{s}"],
        "M1": cat[f"mass_median_{s}"], "M1_err": cat[f"mass_std_{s}"],
        "edad": cat[f"age_median_{s}"], "edad_err": cat[f"age_std_{s}"],
    })
    p["R1"] = radio_sb(p["Teff1"], p["logL1"])
    p["R1_err"] = p["R1"] * np.hypot(np.log(10) / 2 * p["logL1_err"], 2 * p["Teff1_err"] / p["Teff1"])
    partes.append(p)

primarias = pd.concat(partes, ignore_index=True).dropna(subset=["Teff1", "logL1"])
print(f"{len(primarias)} primarias (TIC, rama); por rama:", primarias["rama"].value_counts().to_dict())
primarias.head()

## 4) Secundaria: una fila por (TIC, Sector, rama, malla)

- `R2 = R1·sqrt(Depth_primary)` (de `Depth_primary = (R2/R1)^2`).
- `σ_R2/R2 = ½·σ_D/D`, más `σ_R1/R1` en cuadratura si `PROPAGAR_ERROR_R1`.
- Para cada rama, (R2, edad de la rama) se evalúa contra la **malla propia** de la
  rama y contra **BHAC15**. Se obtienen M2, Teff2 y `L2 = R2^2 (Teff2/T_sun)^4`.
- σ_Teff2 y σ_M2 se calculan por diferencias finitas en R2 ± σ_R2 y en edad ± σ_edad
  (recortada al rango de la malla), igual que en `3_diagramas`, y se suman en
  cuadratura. El resultado es NaN si ninguno de los dos lados cae dentro de la malla.

In [ ]:
def r2_desde_profundidad(R1, depth):
    return R1 * np.sqrt(depth)


def sigma_r2(R2, depth, depth_err, R1, R1_err):
    rel2 = (0.5 * depth_err / depth) ** 2
    if PROPAGAR_ERROR_R1:
        rel2 = rel2 + (R1_err / R1) ** 2
    return R2 * np.sqrt(rel2)


def _semi_diff(hi, lo, mid):
    if np.isfinite(hi) and np.isfinite(lo):
        return (hi - lo) / 2
    if np.isfinite(hi) and np.isfinite(mid):
        return hi - mid
    if np.isfinite(lo) and np.isfinite(mid):
        return mid - lo
    return np.nan


def propiedades_secundaria(malla, R2, R2_err, edad, edad_err):
    c = malla.en_radio(R2, edad)
    dT = dM = [0.0, 0.0]
    if np.isfinite(c["M"]):
        dT, dM = [0.0, 0.0], [0.0, 0.0]
        if np.isfinite(R2_err) and R2_err > 0:
            hi, lo = malla.en_radio(R2 + R2_err, edad), malla.en_radio(max(R2 - R2_err, 1e-4), edad)
            dT[0] = _semi_diff(hi["Teff"], lo["Teff"], c["Teff"])
            dM[0] = _semi_diff(hi["M"], lo["M"], c["M"])
        if np.isfinite(edad_err) and edad_err > 0:
            e_hi = min(edad + edad_err, malla.edad_max)
            e_lo = max(edad - edad_err, malla.edad_min)
            hi, lo = malla.en_radio(R2, e_hi), malla.en_radio(R2, e_lo)
            dT[1] = _semi_diff(hi["Teff"], lo["Teff"], c["Teff"])
            dM[1] = _semi_diff(hi["M"], lo["M"], c["M"])
    return dict(M2=c["M"], Teff2=c["Teff"], L2=lum_sb(c["Teff"], R2),
                Teff2_err=np.hypot(*dT) if np.isfinite(c["M"]) else np.nan,
                M2_err=np.hypot(*dM) if np.isfinite(c["M"]) else np.nan)


def calcular_secundarias(profundidades, claves):
    # profundidades: columnas claves + Depth_primary, Depth_primary_err
    # las filas sin profundidad se conservan (R2 = NaN) para que su primaria siga en las tablas/figuras
    base = profundidades.merge(primarias, on="TIC", how="inner")
    base["R2"] = r2_desde_profundidad(base["R1"], base["Depth_primary"])
    base["R2_err"] = sigma_r2(base["R2"], base["Depth_primary"], base["Depth_primary_err"],
                              base["R1"], base["R1_err"])
    filas = []
    for _, b in base.iterrows():
        for nombre_malla in dict.fromkeys([b["rama"], "BHAC15"]):
            props = propiedades_secundaria(mallas[nombre_malla], b["R2"], b["R2_err"], b["edad"], b["edad_err"])
            filas.append({**{k: b[k] for k in claves}, "rama": b["rama"], "malla": nombre_malla,
                          "Depth_primary": b["Depth_primary"], "Depth_primary_err": b["Depth_primary_err"],
                          "R1": b["R1"], "R2": b["R2"], "R2_err": b["R2_err"],
                          "edad": b["edad"], "edad_err": b["edad_err"], **props})
    return pd.DataFrame(filas)


fuera_de_cat = sorted(set(priors["TIC"]) - set(cat["TIC"]))
if fuera_de_cat:
    print("TICs con priors pero sin ajuste massage en el catálogo (se omiten):", fuera_de_cat)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    secundarias = calcular_secundarias(priors, claves=["TIC", "Sector"])

conteo = (secundarias.assign(ok=secundarias["Teff2"].notna())
          .groupby(["rama", "malla"])["ok"].agg(dentro_de_malla="sum", total="size"))
print(conteo)
secundarias.to_csv(SALIDA / "secundarias_por_sector_largo.csv", index=False)

## 5) Combinación de sectores por TIC

Promedio pesado por inverso de varianza de `Depth_primary` sobre los sectores de
cada TIC:

`D = Σ wᵢDᵢ / Σ wᵢ`, `σ_D = (Σ wᵢ)^(-1/2)`, `wᵢ = 1/σᵢ²`,

y `χ²_red = Σ wᵢ(Dᵢ − D)² / (n − 1)`. Si `χ²_red > CHI2_RED_MAX`, los sectores no
concuerdan entre sí dentro de sus errores (por ejemplo TIC 264741359, con S32 en ~5 % y
los demás sectores en ~23 %). En ese caso el valor combinado no es confiable y conviene
revisar las curvas a mano. Cuando χ²_red > 1, σ_D se infla por sqrt(χ²_red).

In [ ]:
def combinar_sectores(g):
    g = g.dropna(subset=["Depth_primary", "Depth_primary_err"])
    g = g.loc[g["Depth_primary_err"] > 0]
    if g.empty:
        return pd.Series({"Depth_primary": np.nan, "Depth_primary_err": np.nan, "n_sectores": 0,
                          "sectores": "", "chi2_red": np.nan})
    w = 1 / g["Depth_primary_err"] ** 2
    D = np.sum(w * g["Depth_primary"]) / w.sum()
    n = len(g)
    chi2_red = np.sum(w * (g["Depth_primary"] - D) ** 2) / (n - 1) if n > 1 else np.nan
    sD = w.sum() ** -0.5 * (np.sqrt(chi2_red) if n > 1 and chi2_red > 1 else 1.0)
    return pd.Series({"Depth_primary": D, "Depth_primary_err": sD, "n_sectores": n,
                      "sectores": ",".join(str(int(s)) for s in g["Sector"]), "chi2_red": chi2_red})


prof_tic = priors.groupby("TIC").apply(combinar_sectores).reset_index()
prof_tic["inconsistente"] = prof_tic["chi2_red"] > CHI2_RED_MAX

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    secundarias_tic = calcular_secundarias(prof_tic, claves=["TIC"]).merge(
        prof_tic[["TIC", "n_sectores", "sectores", "chi2_red", "inconsistente"]], on="TIC")

secundarias_tic.to_csv(SALIDA / "secundarias_por_tic_largo.csv", index=False)
print("TICs con sectores inconsistentes:", prof_tic.loc[prof_tic["inconsistente"], "TIC"].tolist())
prof_tic

## 6) Tabla resumen (formato ancho)

Una fila por (TIC, Sector), con las mismas columnas que
`resumen_diagramas_hr_por_tic.csv`:

- `_A`: la primaria, tomada de la primera rama de `ORDEN_RAMAS` que tenga Teff y L.
- `_B`: la secundaria "elegida", tomada de la primera rama con solución en
  `MALLA_SECUNDARIA`.
- `_B_BHAC15_<rama>`: la secundaria contra BHAC15, una columna por rama.
- Nuevo: `_B_propia_<rama>`, la secundaria contra la malla de la propia rama (lo que
  antes era `Teff2_MIST`/`Teff2_Somers17`).

In [ ]:
def tabla_ancha(sec_largo, claves):
    filas = []
    for key, g in sec_largo.groupby(claves):
        key = key if isinstance(key, tuple) else (key,)
        fila = dict(zip(claves, key))
        tic = fila["TIC"]

        # -------- primaria elegida --------
        for rama in ORDEN_RAMAS:
            p = primarias.loc[(primarias["TIC"] == tic) & (primarias["rama"] == rama)]
            if len(p):
                p = p.iloc[0]
                fila.update(modelo_A=rama, edad_A_Myr=p["edad"], Teff_A=p["Teff1"], L_A=10 ** p["logL1"],
                            R_A=p["R1"], R_A_err=p["R1_err"], M_A=p["M1"])
                break

        fila["Depth_primary"] = g["Depth_primary"].iloc[0]
        fila["Depth_primary_err"] = g["Depth_primary_err"].iloc[0]

        # -------- todas las combinaciones rama x malla --------
        for _, s in g.iterrows():
            suf = f"B_BHAC15_{s['rama']}" if s["malla"] == "BHAC15" else f"B_propia_{s['rama']}"
            fila.update({f"edad_{suf}_Myr": s["edad"], f"edad_{suf}_err_Myr": s["edad_err"],
                         f"Teff_{suf}": s["Teff2"], f"Teff_{suf}_err": s["Teff2_err"],
                         f"L_{suf}": s["L2"], f"R_{suf}": s["R2"], f"R_{suf}_err": s["R2_err"],
                         f"M_{suf}": s["M2"], f"M_{suf}_err": s["M2_err"]})

        # -------- secundaria elegida --------
        for rama in ORDEN_RAMAS:
            s = g.loc[(g["rama"] == rama) & (g["malla"] == MALLA_SECUNDARIA) & g["Teff2"].notna()]
            if len(s):
                s = s.iloc[0]
                fila.update(modelo_B=rama, edad_B_Myr=s["edad"], Teff_B=s["Teff2"], L_B=s["L2"],
                            R_B=s["R2"], M_B=s["M2"], Teff_B_err=s["Teff2_err"], M_B_err=s["M2_err"])
                break
        filas.append(fila)

    t = pd.DataFrame(filas).merge(cat[["TIC", "OBJECT", "RA", "DEC"]], on="TIC", how="left")
    primero = [*claves, "OBJECT", "RA", "DEC", "Depth_primary", "Depth_primary_err",
               "modelo_A", "edad_A_Myr", "Teff_A", "L_A", "R_A", "R_A_err", "M_A",
               "modelo_B", "edad_B_Myr", "Teff_B", "L_B", "R_B", "M_B", "Teff_B_err", "M_B_err"]
    primero = [c for c in primero if c in t.columns]
    return t[primero + sorted(c for c in t.columns if c not in primero)]


resumen = tabla_ancha(secundarias, ["TIC", "Sector"])
resumen_tic = tabla_ancha(secundarias_tic, ["TIC"]).merge(
    prof_tic[["TIC", "n_sectores", "sectores", "chi2_red", "inconsistente"]], on="TIC")

resumen.to_csv(SALIDA / "resumen_por_tic_sector.csv", index=False)
resumen_tic.to_csv(SALIDA / "resumen_por_tic_combinado.csv", index=False)
print(f"Guardado: resumen_por_tic_sector.csv ({len(resumen)} filas) y "
      f"resumen_por_tic_combinado.csv ({len(resumen_tic)} filas) en {SALIDA}")
resumen_tic[["TIC", "n_sectores", "chi2_red", "inconsistente", "Depth_primary",
             "modelo_A", "Teff_A", "R_A", "M_A", "modelo_B", "Teff_B", "Teff_B_err", "R_B", "M_B", "M_B_err"]]

In [ ]:
# -------- comparación contra la versión anterior (3_diagramas_hr_isocronas) --------
# Con priors_v2 las diferencias son grandes: vienen sobre todo de las profundidades v2
# (R_B = R_A*sqrt(Depth)), no del interpolador.
viejo_path = RUTA_ORION / "resumen_diagramas_hr_por_tic.csv"
if viejo_path.exists():
    viejo = pd.read_csv(viejo_path)
    comp = resumen.merge(viejo, on=["TIC", "Sector"], suffixes=("", "_viejo"))
    for col in ["R_B", "Teff_B", "M_B"]:
        a, b = comp[col], comp[f"{col}_viejo"]
        ambos = a.notna() & b.notna()
        rel = (a[ambos] / b[ambos] - 1).abs()
        print(f"{col:7s}: válidos nuevo={a.notna().sum():2d}  viejo={b.notna().sum():2d}  "
              f"ambos={ambos.sum():2d}  |Δ| relativa mediana={rel.median():.2%}  máx={rel.max():.2%}")

## 6b) Índice de sistemas

Los diagramas HR usan el **mismo índice interno** que `explore_priors_v2.ipynb`: los sistemas se
numeran del 1 al N ordenados por periodo orbital. El periodo de cada sistema es la mediana de
sus sectores válidos; se descartan los sectores `eclipse_ancho` cuando el TIC tiene otros buenos.
El índice se calcula aquí con la misma regla, a partir de `priors_v2.csv`, así que este notebook no
depende del otro. Si existe `figuras_explore_priors/indice_sistemas.csv`, se comprueba que los dos
coincidan.

In [ ]:
def indice_sistemas(pri):
    ancho = pri.get("Calidad_depth", pd.Series("", index=pri.index)).fillna("").str.contains("eclipse_ancho")
    P = {}
    for tic, g in pri.assign(ancho=ancho).groupby("TIC"):
        buenos = g.loc[~g["ancho"]]
        P[tic] = (buenos if len(buenos) else g)["P_orb"].median()
    orden = sorted(P, key=P.get)
    return {int(tic): i + 1 for i, tic in enumerate(orden)}


INDICE = indice_sistemas(priors)
TIC_DE = {v: k for k, v in INDICE.items()}

csv_indice = RUTA_ORION / "figuras_explore_priors" / "indice_sistemas.csv"
if csv_indice.exists():
    ext = dict(zip(pd.read_csv(csv_indice)["TIC"].astype(int), pd.read_csv(csv_indice)["indice"]))
    difieren = {t: (INDICE.get(t), i) for t, i in ext.items() if INDICE.get(t) != i}
    print("Índice consistente con explore_priors_v2." if not difieren else
          f"[!] El índice difiere de explore_priors_v2 en {difieren}: vuelve a correr ese notebook.")
pd.DataFrame({"indice": list(INDICE.values()), "TIC": list(INDICE.keys())}).set_index("indice").T

## 7) Diagramas HR y R–masa por sistema

Curvas: MIST a la edad de MIST (azul), Somers17 a la edad de Somers17 (naranja) y
BHAC15 a cada una de esas dos edades (verde). Puntos: primaria (círculo) y, para
cada sector, la secundaria (triángulo = malla propia, diamante = BHAC15). La
secundaria combinada de todos los sectores se dibuja como estrella hueca con barras
de error. Los límites de los ejes se fijan con los puntos observados y no con la
curva completa.

Cada figura se identifica con el índice del sistema (sección 6b) y se guarda como
`sistema_NN_hr_rm.png`. Un TIC del catálogo de massage que no tiene curva de luz analizada
(TIC 427393298) no entra en el índice; su figura se guarda como `sin_indice_TIC_<tic>_hr_rm.png`.

In [ ]:
COLOR_RAMA = {"MIST": "tab:blue", "Somers17": "tab:orange"}
PAD = 4.0

dir_tic = SALIDA / "hr_por_tic"
dir_tic.mkdir(exist_ok=True)


def figura_tic(tic):
    prim = primarias.loc[primarias["TIC"] == tic]
    if prim["edad"].isna().all():
        print(f"Sistema {INDICE.get(tic)} (TIC {tic}): sin edad en ninguna rama, se omite.")
        return
    fig, (ax_hr, ax_rm) = plt.subplots(1, 2, figsize=(13, 5.5))
    pts_T, pts_L, pts_M, pts_R = [], [], [], []

    # -------- isocronas --------
    for _, p in prim.iterrows():
        for nombre_malla, ls in ((p["rama"], "-"), ("BHAC15", "--" if p["rama"] == "MIST" else ":")):
            iso = mallas[nombre_malla].isocrona(p["edad"])
            if iso is None:
                continue
            color = "tab:green" if nombre_malla == "BHAC15" else COLOR_RAMA[p["rama"]]
            etiqueta = (f"{nombre_malla}, {p['edad']:.3g} Myr"
                        + (f" (edad {p['rama']})" if nombre_malla == "BHAC15" else ""))
            ax_hr.plot(10 ** iso["logTeff"], 10 ** iso["logL"], color=color, ls=ls, label=etiqueta)
            ax_rm.plot(iso["mass"], 10 ** iso["logR"], color=color, ls=ls)

    # -------- primarias --------
    for _, p in prim.iterrows():
        c = COLOR_RAMA[p["rama"]]
        ax_hr.scatter(p["Teff1"], 10 ** p["logL1"], color=c, edgecolor="k", s=90, zorder=5,
                      label=f"primaria ({p['rama']})")
        pts_T.append(p["Teff1"]); pts_L.append(10 ** p["logL1"])
        if np.isfinite(p["M1"]):
            ax_rm.scatter(p["M1"], p["R1"], color=c, edgecolor="k", s=90, zorder=5)
            pts_M.append(p["M1"]); pts_R.append(p["R1"])

    # -------- secundarias por sector --------
    for _, s in secundarias.loc[(secundarias["TIC"] == tic) & secundarias["Teff2"].notna()].iterrows():
        c = "tab:green" if s["malla"] == "BHAC15" else COLOR_RAMA[s["rama"]]
        mk = "D" if s["malla"] == "BHAC15" else "^"
        et = f"secundaria ({s['malla']}" + (f"/{s['rama']})" if s["malla"] == "BHAC15" else ")")
        ax_hr.scatter(s["Teff2"], s["L2"], color=c, marker=mk, edgecolor="k", s=60, zorder=6, label=et)
        ax_hr.annotate(f"S{int(s['Sector'])}", (s["Teff2"], s["L2"]), xytext=(4, 3),
                       textcoords="offset points", fontsize=6.5, alpha=0.8)
        pts_T.append(s["Teff2"]); pts_L.append(s["L2"])
        if np.isfinite(s["M2"]):
            ax_rm.scatter(s["M2"], s["R2"], color=c, marker=mk, edgecolor="k", s=60, zorder=6)
            pts_M.append(s["M2"]); pts_R.append(s["R2"])

    # -------- secundaria combinada (malla elegida) --------
    for _, s in secundarias_tic.loc[(secundarias_tic["TIC"] == tic) & (secundarias_tic["malla"] == MALLA_SECUNDARIA)
                                    & secundarias_tic["Teff2"].notna()].iterrows():
        c = COLOR_RAMA[s["rama"]]
        ax_hr.errorbar(s["Teff2"], s["L2"], xerr=s["Teff2_err"], fmt="*", ms=15, mfc="none", mec=c,
                       ecolor=c, zorder=7, label=f"secundaria combinada ({MALLA_SECUNDARIA}/{s['rama']})")
        if np.isfinite(s["M2"]):
            ax_rm.errorbar(s["M2"], s["R2"], xerr=s["M2_err"], yerr=s["R2_err"], fmt="*", ms=15,
                           mfc="none", mec=c, ecolor=c, zorder=7)

    # -------- estilo --------
    for ax in (ax_hr, ax_rm):
        ax.set_xscale("log"); ax.set_yscale("log"); ax.grid(alpha=0.25, ls="--")
    if pts_T:
        ax_hr.set_xlim(max(pts_T) * PAD, min(pts_T) / PAD)   # Teff alta a la izquierda
        ax_hr.set_ylim(min(pts_L) / PAD, max(pts_L) * PAD)
    if pts_M:
        ax_rm.set_xlim(min(pts_M) / PAD, max(pts_M) * PAD)
        ax_rm.set_ylim(min(pts_R) / PAD, max(pts_R) * PAD)
    idx = INDICE.get(tic)
    nombre = f"Sistema {idx}" if idx else "Sin índice (sin curva de luz)"
    ax_hr.set(xlabel="Teff [K]", ylabel="L [Lsun]", title=f"{nombre} -- diagrama HR")
    ax_rm.set(xlabel="Masa [Msun]", ylabel="R [Rsun]", title=f"{nombre} -- R vs masa")

    h, l = ax_hr.get_legend_handles_labels()
    unicos = dict(zip(l, h))
    fig.legend(unicos.values(), unicos.keys(), loc="lower center", ncol=3, fontsize=8,
               bbox_to_anchor=(0.5, -0.1))
    fig.tight_layout(rect=[0, 0.05, 1, 1])
    archivo = f"sistema_{idx:02d}_hr_rm.png" if idx else f"sin_indice_TIC_{tic}_hr_rm.png"
    fig.savefig(dir_tic / archivo, dpi=200, bbox_inches="tight")
    plt.show() if MOSTRAR_FIGURAS else None
    plt.close(fig)


for tic in sorted(primarias["TIC"].unique(), key=lambda t: INDICE.get(int(t), 999)):
    figura_tic(int(tic))

## 8) Diagramas HR por sector

En cada sector se dibujan todas las estrellas (primaria A y secundaria B elegidas
según la tabla resumen). Hay dos versiones:

- `color_por="Teff"`: estilo divulgación, con ejes log-log en Teff y L, color por
  temperatura y tamaño por radio.
- `color_por="masa"`: log Teff vs log L, color por masa en escala log y tamaño por
  radio. `*` es la primaria y `o` la secundaria; en gris, las que no tienen masa.

Cada estrella se etiqueta con el índice del sistema más A (primaria) o B (secundaria);
por ejemplo, `8A` y `8B` son las dos estrellas del sistema 8.

In [ ]:
dir_sector = SALIDA / "hr_por_sector"
dir_sector.mkdir(exist_ok=True)


def estrellas_del_sector(sector):
    r = resumen.loc[resumen["Sector"] == sector]
    a = r.dropna(subset=["Teff_A"]).drop_duplicates("TIC")
    b = r.dropna(subset=["Teff_B"])
    return pd.concat([
        pd.DataFrame({"tic": a["TIC"], "comp": "A", "Teff": a["Teff_A"], "L": a["L_A"], "R": a["R_A"], "M": a["M_A"]}),
        pd.DataFrame({"tic": b["TIC"], "comp": "B", "Teff": b["Teff_B"], "L": b["L_B"], "R": b["R_B"], "M": b["M_B"]}),
    ], ignore_index=True)


def tamano(R):
    return 25 + 55 * np.log10(np.clip(R, 1e-2, None) / 1e-2)


def etiquetar(ax, xs, ys, textos, sep_px=14):
    # coloca cada etiqueta en la primera posición libre (lejos de puntos y de etiquetas ya puestas)
    ax.figure.canvas.draw()
    pts = ax.transData.transform(np.column_stack([xs, ys]))
    ocupado = [tuple(q) for q in pts]
    opciones = [(7, 6), (7, -13), (-9, 6), (-9, -13), (0, 12), (0, -18), (14, -3), (-16, -3),
                (12, 14), (-14, 14), (12, -20), (-14, -20)]
    for (px, py), txt in zip(pts, textos):
        for dx, dy in opciones:
            cx, cy = px + dx + (5 if dx >= 0 else -5), py + dy + 4
            if all(np.hypot(cx - ox, cy - oy) > sep_px for ox, oy in ocupado):
                break
        ax.annotate(txt, ax.transData.inverted().transform((px, py)), xytext=(dx, dy),
                    textcoords="offset points", fontsize=8, ha="left" if dx >= 0 else "right", zorder=6,
                    arrowprops=dict(arrowstyle="-", lw=0.4, color="0.5") if abs(dx) > 10 or abs(dy) > 13 else None)
        ocupado.append((cx, cy))


def figura_sector(sector, color_por="masa", mostrar_nombres=True):
    df = estrellas_del_sector(sector)
    if df.empty:
        print(f"Sector {sector}: sin estrellas con datos suficientes.")
        return
    fig, ax = plt.subplots(figsize=(9, 7))
    x = np.log10(df["Teff"]) if color_por == "masa" else df["Teff"]
    y = np.log10(df["L"]) if color_por == "masa" else df["L"]

    if color_por == "Teff":
        cmap = plt.get_cmap("RdYlBu")
        norm = plt.Normalize(df["Teff"].min(), df["Teff"].max())
        ax.scatter(x, y, s=tamano(df["R"]), c=cmap(norm(df["Teff"])), edgecolor="k", lw=0.4, alpha=0.9, zorder=5)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set(xlabel="Temperatura efectiva (K)", ylabel="Luminosidad (luminosidades solares)",
               title=f"Diagrama Hertzsprung-Russell -- Sector {sector}")
    else:
        con_masa = df["M"].notna()
        norm = LogNorm(df.loc[con_masa, "M"].min(), df.loc[con_masa, "M"].max()) if con_masa.any() else None
        for comp, (mk, fac) in {"A": ("*", 1.8), "B": ("o", 1.0)}.items():
            m = (df["comp"] == comp) & con_masa
            if m.any():
                ax.scatter(x[m], y[m], s=tamano(df.loc[m, "R"]) * fac, c=df.loc[m, "M"], cmap="viridis",
                           norm=norm, marker=mk, edgecolor="k", lw=0.4, alpha=0.9, zorder=5)
            m = (df["comp"] == comp) & ~con_masa
            if m.any():
                ax.scatter(x[m], y[m], s=tamano(df.loc[m, "R"]) * fac, color="0.75", marker=mk,
                           edgecolor="k", lw=0.4, alpha=0.9, zorder=4)
        if norm is not None:
            sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
            cb = fig.colorbar(sm, ax=ax, pad=0.02)
            ticks = np.geomspace(norm.vmin, norm.vmax, 5)
            cb.set_ticks(ticks); cb.set_ticklabels([f"{t:.2g}" for t in ticks])
            cb.set_label(r"Masa (M$_\odot$)")
        forma = [plt.scatter([], [], marker="*", s=140, color="0.6", edgecolor="k", label="Primaria"),
                 plt.scatter([], [], marker="o", s=80, color="0.6", edgecolor="k", label="Secundaria")]
        if (~con_masa).any():
            forma.append(plt.scatter([], [], marker="s", s=80, color="0.75", edgecolor="k", label="Sin masa estimada"))
        ax.add_artist(ax.legend(handles=forma, loc="upper right", fontsize=8, frameon=False))
        ax.set(xlabel="log Teff [K]", ylabel=r"log (L / L$_\odot$)",
               title=f"Diagrama HR (log Te vs log L) -- Sector {sector}")

    if mostrar_nombres:
        ax.invert_xaxis()   # antes de etiquetar, para que las posiciones en pantalla sean las finales
        etiquetar(ax, np.asarray(x, float), np.asarray(y, float),
                  [f"{INDICE.get(int(t), '?')}{c}" for t, c in zip(df["tic"], df["comp"])])
        ax.invert_xaxis()   # se deshace: el eje se invierte de nuevo más abajo

    radios = [r for r in [10, 1, 0.1, 0.01] if df["R"].min() / 3 <= r <= df["R"].max() * 3] or [1]
    h = [plt.scatter([], [], s=tamano(r), color="0.6", edgecolor="k", lw=0.4, label=f"{r:g}") for r in radios]
    ax.legend(handles=h, title="Radio\n(radios solares)", loc="lower left", fontsize=8,
              title_fontsize=8, labelspacing=1.3, frameon=False)
    ax.invert_xaxis()
    ax.grid(alpha=0.2, ls="--")
    fig.tight_layout()
    nombre = f"HR_sector_{sector}.png" if color_por == "Teff" else f"HR_logTL_masa_sector_{sector}.png"
    fig.savefig(dir_sector / nombre, dpi=200, bbox_inches="tight")
    plt.show() if MOSTRAR_FIGURAS else None
    plt.close(fig)


for sector in sorted(resumen["Sector"].unique()):
    figura_sector(sector, color_por="Teff")
    figura_sector(sector, color_por="masa")

## Archivos generados (en `resultados_pipeline/`)

| archivo | contenido |
|---|---|
| `secundarias_por_sector_largo.csv` | una fila por (TIC, Sector, rama, malla) |
| `secundarias_por_tic_largo.csv` | igual, con la profundidad combinada de todos los sectores |
| `resumen_por_tic_sector.csv` | tabla ancha, compatible con `resumen_diagramas_hr_por_tic.csv` |
| `resumen_por_tic_combinado.csv` | tabla ancha, una fila por TIC, con `chi2_red` e `inconsistente` |
| `hr_por_tic/`, `hr_por_sector/` | figuras |
| `priors_recalculados.csv`, `curvas_de_luz/` | solo si `RECALCULAR_PRIORS = True` |